In [1]:
pip install numpy

   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
    --------------------------------------- 0.3/12.4 MB ? eta -:--:--
    --------------------------------------- 0.3/12.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.4 MB 709.4 kB/s eta 0:00:17
   - -------------------------------------- 0.5/12.4 MB 709.4 kB/s eta 0:00:17
   - -------------------------------------- 0.5/12.4 MB 709.4 kB/s eta 0:00:17
   - -------------------------------------- 0.5/12.4 MB 709.4 kB/s eta 0:00:17
   -- ------------------------------------- 0.8/12.4 MB 426.3 kB/s eta 0:00:28
   -- ------------------------------------- 0.8/12.4 MB 426.3 kB/s eta 0:00:28
   -- ------------------------------------- 0.8/12.4 MB 426.3 kB/s eta 0:00:28
   --- ------------------------------------ 1.0/12.4 MB 428.2 kB/s eta 0:00:27
   --- ------------------------------------ 1.0/12.4 MB 428.2 kB/s eta 0:00:27
   --- ------------------------------------ 1.0/12.4 MB 428.2 kB/s eta 0:00:27



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import numpy as np
print(np.version)

<module 'numpy.version' from 'C:\\Users\\KMGWALANI\\AppData\\Local\\Programs\\Python\\Python314\\Lib\\site-packages\\numpy\\version.py'>


In [2]:
# Import required modules
import pickle              # Used to save and load Python objects from files
import os                  # Used to check if files already exist
from datetime import datetime   # Used to store date and time of transactions
import numpy as np         # Used to calculate summary statistics
import hashlib             # Used to hash passwords for login functionality


# File names where account and transaction data will be stored
ACCOUNTS_FILE = "accounts.pkl"
TRANSACTIONS_FILE = "transactions.pkl"


# Function to load data from a pickle file
def load_pickle_file(filename, default_value):
    # Check if file exists
    if os.path.exists(filename):
        try:
            # Open file in binary read mode and load data
            with open(filename, "rb") as file:
                return pickle.load(file)
        except (EOFError, pickle.UnpicklingError):
            # If file is empty or corrupted, return default value
            return default_value
    # If file does not exist, return default value
    return default_value


# Function to save data into a pickle file
def save_pickle_file(filename, data):
    with open(filename, "wb") as file:
        pickle.dump(data, file)


# Load existing account and transaction data when program starts
accounts = load_pickle_file(ACCOUNTS_FILE, {})
transactions = load_pickle_file(TRANSACTIONS_FILE, [])


# Class representing a single bank account
class BankAccount:
    def __init__(self, holder_name, account_number, account_type, balance=0.0, username=None, password_hash=None):
        self.holder_name = holder_name
        self.account_number = account_number
        self.account_type = account_type
        self.balance = float(balance)
        self.username = username
        self.password_hash = password_hash

    # Method to deposit money into account
    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit amount must be positive.")
        self.balance += amount

    # Method to withdraw money from account
    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive.")
        if amount > self.balance:
            raise ValueError("Insufficient balance.")
        self.balance -= amount

    # Convert object into dictionary format so it can be stored in pickle file
    def to_dict(self):
        return {
            "holder_name": self.holder_name,
            "account_number": self.account_number,
            "account_type": self.account_type,
            "balance": self.balance,
            "username": self.username,
            "password_hash": self.password_hash,
        }


# Class that manages the complete banking system
class BankSystem:
    def __init__(self):
        # Load accounts and transactions from files
        self.accounts = self.load_accounts()
        self.transactions = self.load_transactions()

    # Load accounts from file and convert dictionaries back into BankAccount objects
    def load_accounts(self):
        raw_accounts = load_pickle_file(ACCOUNTS_FILE, {})
        loaded_accounts = {}
        for account_number, data in raw_accounts.items():
            loaded_accounts[account_number] = BankAccount(**data)
        return loaded_accounts

    # Save all account objects into file after converting them into dictionaries
    def save_accounts(self):
        serializable_accounts = {
            account_number: account.to_dict()
            for account_number, account in self.accounts.items()
        }
        save_pickle_file(ACCOUNTS_FILE, serializable_accounts)

    # Load transaction history from file
    def load_transactions(self):
        return load_pickle_file(TRANSACTIONS_FILE, [])

    # Save transaction history to file
    def save_transactions(self):
        save_pickle_file(TRANSACTIONS_FILE, self.transactions)

    # Auto-generate account number
    def generate_account_number(self):
        if not self.accounts:
            return 1001
        return max(self.accounts.keys()) + 1

    # Hash password before storing it
    def hash_password(self, password):
        return hashlib.sha256(password.encode()).hexdigest()

    # Create a new bank account
    def create_account(self, holder_name, account_type, initial_balance, username=None, password=None):
        if initial_balance < 0:
            raise ValueError("Initial balance cannot be negative.")
        account_number = self.generate_account_number()
        password_hash = self.hash_password(password) if password else None

        new_account = BankAccount(
            holder_name,
            account_number,
            account_type,
            initial_balance,
            username,
            password_hash,
        )

        self.accounts[account_number] = new_account

        # Record opening transaction
        self.record_transaction(account_number, "Account Opened", initial_balance)
        return account_number

    # Bonus: login authentication
    def authenticate(self, username, password):
        password_hash = self.hash_password(password)
        for account in self.accounts.values():
            if account.username == username and account.password_hash == password_hash:
                return account
        return None

    # Find an account by account number
    def get_account(self, account_number):
        if account_number not in self.accounts:
            raise ValueError("Account does not exist.")
        return self.accounts[account_number]

    # Store transaction details into transaction list
    def record_transaction(self, account_number, transaction_type, amount, note=""):
        self.transactions.append(
            {
                "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "account_number": account_number,
                "type": transaction_type,
                "amount": float(amount),
                "note": note,
            }
        )

    # Deposit money
    def deposit(self, account_number, amount):
        account = self.get_account(account_number)
        account.deposit(amount)
        self.record_transaction(account_number, "Deposit", amount)

    # Withdraw money
    def withdraw(self, account_number, amount):
        account = self.get_account(account_number)
        account.withdraw(amount)
        self.record_transaction(account_number, "Withdrawal", amount)

    # Transfer money from one account to another
    def transfer(self, from_account_number, to_account_number, amount):
        if from_account_number == to_account_number:
            raise ValueError("Cannot transfer to the same account.")

        sender = self.get_account(from_account_number)
        receiver = self.get_account(to_account_number)

        sender.withdraw(amount)
        receiver.deposit(amount)

        self.record_transaction(from_account_number, "Transfer Out", amount, f"To Account {to_account_number}")
        self.record_transaction(to_account_number, "Transfer In", amount, f"From Account {from_account_number}")

    # Get transaction history of a specific account
    def get_transaction_history(self, account_number):
        self.get_account(account_number)
        return [
            transaction
            for transaction in self.transactions
            if transaction["account_number"] == account_number
        ]

    # Generate summary statistics using NumPy
    def get_summary_statistics(self, account_number):
        history = self.get_transaction_history(account_number)

        # Collect deposits and withdrawals separately
        deposits = np.array(
            [t["amount"] for t in history if t["type"] in ["Deposit", "Transfer In", "Account Opened"]],
            dtype=float
        )
        withdrawals = np.array(
            [t["amount"] for t in history if t["type"] in ["Withdrawal", "Transfer Out"]],
            dtype=float
        )
        all_amounts = np.array([t["amount"] for t in history], dtype=float)

        # Calculate total and average values
        total_deposits = float(np.sum(deposits)) if deposits.size else 0.0
        total_withdrawals = float(np.sum(withdrawals)) if withdrawals.size else 0.0
        average_transaction = float(np.mean(all_amounts)) if all_amounts.size else 0.0

        return {
            "total_deposits": total_deposits,
            "total_withdrawals": total_withdrawals,
            "average_transaction": average_transaction,
        }

    # Display account details
    def display_account_details(self, account_number):
        account = self.get_account(account_number)
        print("\nAccount Details")
        print(f"Holder Name   : {account.holder_name}")
        print(f"Account Number: {account.account_number}")
        print(f"Account Type  : {account.account_type}")
        print(f"Current Balance: {account.balance:.2f}")

    # Display full transaction history and summary
    def display_transaction_history(self, account_number):
        history = self.get_transaction_history(account_number)

        if not history:
            print("\nNo transactions found for this account.")
            return

        print("\nTransaction History")
        for transaction in history:
            note = f" | {transaction['note']}" if transaction['note'] else ""
            print(f"{transaction['date']} | {transaction['type']} | Amount: {transaction['amount']:.2f}{note}")

        stats = self.get_summary_statistics(account_number)
        print("\nSummary Statistics")
        print(f"Total Deposits    : {stats['total_deposits']:.2f}")
        print(f"Total Withdrawals : {stats['total_withdrawals']:.2f}")
        print(f"Average Transaction: {stats['average_transaction']:.2f}")

    # Save everything before program exits
    def save_all(self):
        self.save_accounts()
        self.save_transactions()


# Function to ensure amount entered is positive
def get_positive_amount(prompt):
    while True:
        try:
            amount = float(input(prompt))
            if amount <= 0:
                print("Amount must be greater than zero.")
                continue
            return amount
        except ValueError:
            print("Please enter a valid numeric amount.")


# Function to safely take account number input
def get_account_number_input(prompt):
    while True:
        try:
            return int(input(prompt))
        except ValueError:
            print("Please enter a valid account number.")


# Main menu-driven program
def main_menu():
    bank = BankSystem()

    while True:
        print("\n=== Bank Account Management System ===")
        print("1. Open a new account")
        print("2. View account details")
        print("3. Deposit money")
        print("4. Withdraw money")
        print("5. Transfer money")
        print("6. View transaction history")
        print("7. Login (Bonus)")
        print("8. Exit")

        choice = input("Enter your choice: ").strip()

        try:
            if choice == "1":
                holder_name = input("Enter account holder name: ").strip()
                account_type = input("Enter account type (Savings/Current): ").strip().capitalize()
                initial_balance = get_positive_amount("Enter initial balance: ")

                create_login = input("Create login credentials? (y/n): ").strip().lower()
                username = None
                password = None

                if create_login == "y":
                    username = input("Choose username: ").strip()
                    password = input("Choose password: ").strip()

                account_number = bank.create_account(holder_name, account_type, initial_balance, username, password)
                print(f"Account created successfully. Your account number is {account_number}.")

            elif choice == "2":
                account_number = get_account_number_input("Enter account number: ")
                bank.display_account_details(account_number)

            elif choice == "3":
                account_number = get_account_number_input("Enter account number: ")
                amount = get_positive_amount("Enter deposit amount: ")
                bank.deposit(account_number, amount)
                print("Deposit successful.")

            elif choice == "4":
                account_number = get_account_number_input("Enter account number: ")
                amount = get_positive_amount("Enter withdrawal amount: ")
                bank.withdraw(account_number, amount)
                print("Withdrawal successful.")

            elif choice == "5":111
                from_account = get_account_number_input("Enter sender account number: ")
                to_account = get_account_number_input("Enter receiver account number: ")
                amount = get_positive_amount("Enter transfer amount: ")
                bank.transfer(from_account, to_account, amount)
                print("Transfer successful.")

            elif choice == "6":
                account_number = get_account_number_input("Enter account number: ")
                bank.display_transaction_history(account_number)

            elif choice == "7":
                username = input("Enter username: ").strip()
                password = input("Enter password: ").strip()

                account = bank.authenticate(username, password)
                if account:
                    print(f"Login successful. Welcome, {account.holder_name}.")
                    bank.display_account_details(account.account_number)
                else:
                    print("Invalid username or password.")

            elif choice == "8":
                bank.save_all()
                print("Data saved successfully. Exiting program.")
                break

            else:
                print("Invalid choice. Please select a valid menu option.")

        except ValueError as error:
            print(f"Error: {error}")
        except Exception as error:
            print(f"Unexpected error: {error}")


# Program starts from here
if __name__ == "__main__":
    main_menu()



=== Bank Account Management System ===
1. Open a new account
2. View account details
3. Deposit money
4. Withdraw money
5. Transfer money
6. View transaction history
7. Login (Bonus)
8. Exit


Enter your choice:  1
Enter account holder name:  Yash
Enter account type (Savings/Current):  Savings
Enter initial balance:  5000
Create login credentials? (y/n):  y
Choose username:  yash123
Choose password:  pass123


Account created successfully. Your account number is 1001.

=== Bank Account Management System ===
1. Open a new account
2. View account details
3. Deposit money
4. Withdraw money
5. Transfer money
6. View transaction history
7. Login (Bonus)
8. Exit


Enter your choice:  2
Enter account number:  1001



Account Details
Holder Name   : Yash
Account Number: 1001
Account Type  : Savings
Current Balance: 5000.00

=== Bank Account Management System ===
1. Open a new account
2. View account details
3. Deposit money
4. Withdraw money
5. Transfer money
6. View transaction history
7. Login (Bonus)
8. Exit


...


Transaction History
2026-05-10 11:10:59 | Account Opened | Amount: 5000.00
2026-05-10 11:12:04 | Deposit | Amount: 2000.00
2026-05-10 11:12:17 | Withdrawal | Amount: 1000.00
2026-05-10 11:13:29 | Transfer Out | Amount: 500.00 | To Account 1002

Summary Statistics
Total Deposits    : 7000.00
Total Withdrawals : 1500.00
Average Transaction: 2125.00

=== Bank Account Management System ===
1. Open a new account
2. View account details
3. Deposit money
4. Withdraw money
5. Transfer money
6. View transaction history
7. Login (Bonus)
8. Exit


Enter your choice:  7
Enter username:  yash123
Enter password:  pass123


Login successful. Welcome, Yash.

Account Details
Holder Name   : Yash
Account Number: 1001
Account Type  : Savings
Current Balance: 5500.00

=== Bank Account Management System ===
1. Open a new account
2. View account details
3. Deposit money
4. Withdraw money
5. Transfer money
6. View transaction history
7. Login (Bonus)
8. Exit


Enter your choice:  8


Data saved successfully. Exiting program.
